## Random Forest Classification

This notebook trains a Random Forest classifier to predict whether an engine is likely to fail soon.

The model is evaluated on an engine-wise validation set.

The same feature engineering, train-validation split, and class imbalance handling used in the Logistic Regression experiment are used here so that the models can be compared fairly.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [2]:
# Load FD001 training data

columns = (
    ["engine_id", "cycle"]
    + ["setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train = pd.read_csv(
    "CMAPSSData/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns
)

# Create RUL
train["RUL"] = (
    train.groupby("engine_id")["cycle"]
    .transform("max") - train["cycle"]
)

train.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187


In [3]:
# Remove sensors that do not change

constant_sensors = [
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

train_clean = train.drop(columns=constant_sensors)

print("Original shape:", train.shape)
print("Cleaned shape:", train_clean.shape)

Original shape: (20631, 27)
Cleaned shape: (20631, 21)


In [4]:
# Sensor feature engineering

feature_sensors = [
    col for col in train_clean.columns
    if col.startswith("sensor_")
]

feature_data = train_clean.copy()

for sensor in feature_sensors:

    # Rolling mean
    feature_data[f"{sensor}_rolling_mean"] = (
        feature_data.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(window=5, min_periods=1).mean()
        )
    )

    # Rolling standard deviation
    feature_data[f"{sensor}_rolling_std"] = (
        feature_data.groupby("engine_id")[sensor]
        .transform(
            lambda x: x.rolling(window=5, min_periods=1).std()
        )
    )

    # Change from previous cycle
    feature_data[f"{sensor}_delta"] = (
        feature_data.groupby("engine_id")[sensor].diff()
    )

print("Feature data shape:", feature_data.shape)

Feature data shape: (20631, 66)


In [7]:
# Create binary target: failure soon

failure_threshold = 30

feature_data["failure_soon"] = (
    feature_data["RUL"] <= failure_threshold
).astype(int)

print(feature_data["failure_soon"].value_counts())

failure_soon
0    17531
1     3100
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

# Get unique engine IDs
engine_ids = feature_data["engine_id"].unique()

# Split engines, not individual rows
train_engines, val_engines = train_test_split(
    engine_ids,
    test_size=0.20,
    random_state=42
)

# Create train and validation datasets
train_data = feature_data[
    feature_data["engine_id"].isin(train_engines)
].copy()

val_data = feature_data[
    feature_data["engine_id"].isin(val_engines)
].copy()

print("Training engines:", train_data["engine_id"].nunique())
print("Validation engines:", val_data["engine_id"].nunique())

print("Training shape:", train_data.shape)
print("Validation shape:", val_data.shape)

Training engines: 80
Validation engines: 20
Training shape: (16561, 67)
Validation shape: (4070, 67)


In [9]:
# Separate features (X) and target (y)

X_train = train_data.drop(columns=["engine_id", "RUL", "failure_soon"])
y_train = train_data["failure_soon"]

X_val = val_data.drop(columns=["engine_id", "RUL", "failure_soon"])
y_val = val_data["failure_soon"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

X_train shape: (16561, 64)
y_train shape: (16561,)
X_val shape: (4070, 64)
y_val shape: (4070,)


## Handling Missing Values

Rolling standard deviation and delta features can create missing values at the beginning of each engine's operating history.

We use median imputation to handle these missing values. The imputer is fitted only on the training data to avoid data leakage.

In [10]:
import numpy as np

print("NaN in X_train:")
print(X_train.isna().sum().sum())

print("NaN in X_val:")
print(X_val.isna().sum().sum())

NaN in X_train:
2400
NaN in X_val:
600


In [11]:
# Median imputation
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

print("Training shape:", X_train_imputed.shape)
print("Validation shape:", X_val_imputed.shape)

Training shape: (16561, 64)
Validation shape: (4070, 64)


## Random Forest Classifier

Random Forest is trained to predict whether an engine is likely to fail soon.

Class-weight balancing is used because the failure-soon class is smaller than the normal class.

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train_imputed, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [13]:
y_pred_rf = rf_clf.predict(X_val_imputed)

print("First 20 actual values:")
print(y_val.values[:20])

print("\nFirst 20 predicted values:")
print(y_pred_rf[:20])

First 20 actual values:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

First 20 predicted values:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [14]:
from sklearn.metrics import confusion_matrix, classification_report

cm_rf = confusion_matrix(y_val, y_pred_rf)

print("Confusion Matrix:")
print(cm_rf)

print("\nClassification Report:")
print(classification_report(y_val, y_pred_rf))

Confusion Matrix:
[[3326  124]
 [  46  574]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98      3450
           1       0.82      0.93      0.87       620

    accuracy                           0.96      4070
   macro avg       0.90      0.94      0.92      4070
weighted avg       0.96      0.96      0.96      4070



In [15]:
# Get failure probabilities

y_prob_rf = rf_clf.predict_proba(X_val_imputed)[:, 1]

print("First 20 failure probabilities:")
print(y_prob_rf[:20])

First 20 failure probabilities:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [16]:
# test different thresholds
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.91, 0.05)

results_rf = []

for threshold in thresholds:

    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    precision = precision_score(y_val, y_pred_threshold)
    recall = recall_score(y_val, y_pred_threshold)
    f1 = f1_score(y_val, y_pred_threshold)

    results_rf.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

threshold_df_rf = pd.DataFrame(results_rf)

threshold_df_rf

,threshold,precision,recall,f1_score
0,0.10,0.559091,0.991935,0.715116
1,0.15,0.599218,0.988710,0.746196
2,0.20,0.639035,0.982258,0.774317
3,0.25,0.665563,0.972581,0.790301
4,0.30,0.696864,0.967742,0.810263
5,0.35,0.724180,0.961290,0.826057
6,0.40,0.760618,0.953226,0.846099
7,0.45,0.800819,0.946774,0.867701
8,0.50,0.822350,0.925806,0.871017
9,0.55,0.846737,0.900000,0.872557


In [17]:
cost_fp = 1
cost_fn = 5

In [18]:
cost_results_rf = []

for threshold in thresholds:

    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    cm = confusion_matrix(y_val, y_pred_threshold)

    tn, fp, fn, tp = cm.ravel()

    total_cost = (fp * cost_fp) + (fn * cost_fn)

    cost_results_rf.append({
        "threshold": threshold,
        "FP": fp,
        "FN": fn,
        "total_cost": total_cost
    })

cost_df_rf = pd.DataFrame(cost_results_rf)

cost_df_rf

,threshold,FP,FN,total_cost
0,0.10,485,5,510
1,0.15,410,7,445
2,0.20,344,11,399
3,0.25,303,17,388
4,0.30,261,20,361
5,0.35,227,24,347
6,0.40,186,29,331
7,0.45,146,33,311
8,0.50,124,46,354
9,0.55,101,62,411


In [19]:
best_threshold_rf = cost_df_rf.loc[
    cost_df_rf["total_cost"].idxmin(),
    "threshold"
]

min_cost_rf = cost_df_rf["total_cost"].min()

print("Best cost-sensitive threshold:", best_threshold_rf)
print("Minimum total cost:", min_cost_rf)

Best cost-sensitive threshold: 0.45000000000000007
Minimum total cost: 311


In [20]:
y_pred_rf_final = (y_prob_rf >= best_threshold_rf).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

cm_rf_final = confusion_matrix(y_val, y_pred_rf_final)

print("Final Random Forest Confusion Matrix:")
print(cm_rf_final)

print("\nFinal Random Forest Classification Report:")
print(classification_report(y_val, y_pred_rf_final))

Final Random Forest Confusion Matrix:
[[3304  146]
 [  33  587]]

Final Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.97      3450
           1       0.80      0.95      0.87       620

    accuracy                           0.96      4070
   macro avg       0.90      0.95      0.92      4070
weighted avg       0.96      0.96      0.96      4070

